# Задание по Python DA: Cleaning. Zahoruiko Anna

## 1. Очистка и подготовка данных

In [1]:
# импорт библиотек

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
# загрузка файлов 

df_contacts = pd.read_excel('Contacts (Done).xlsx')
df_calls = pd.read_excel('Calls (Done).xlsx')
df_spend = pd.read_excel('Spend (Done).xlsx')
df_deals = pd.read_excel('Deals (Done).xlsx')

### 1.1. Анализ датасета Contacts

In [3]:
print(df_contacts.head())

                    Id Contact Owner Name      Created Time     Modified Time
0  5805028000000645014       Rachel White  27.06.2023 11:28  22.12.2023 13:34
1  5805028000000872003      Charlie Davis  03.07.2023 11:31  21.05.2024 10:23
2  5805028000000889001          Bob Brown  02.07.2023 22:37  21.12.2023 13:17
3  5805028000000907006          Bob Brown  03.07.2023 05:44  29.12.2023 15:20
4  5805028000000939010         Nina Scott  04.07.2023 10:11  16.04.2024 16:14


In [4]:
df_contacts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18548 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Id                  18548 non-null  int64 
 1   Contact Owner Name  18548 non-null  object
 2   Created Time        18548 non-null  object
 3   Modified Time       18548 non-null  object
dtypes: int64(1), object(3)
memory usage: 579.8+ KB


In [5]:
df_contacts.describe()

,Id
count,1.854800e+04
mean,5.805028e+18
std,1.566305e+07
min,5.805028e+18
25%,5.805028e+18
50%,5.805028e+18
75%,5.805028e+18
max,5.805028e+18


In [6]:
df_contacts.isna()

,Id,Contact Owner Name,Created Time,Modified Time
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False
...,...,...,...,...
18543,False,False,False,False
18544,False,False,False,False
18545,False,False,False,False
18546,False,False,False,False


#### **Анализ и предложения по типам данных:**

- **Id - Contact_ID**: int64 - преобразовать в string
- **Created_Time**: object - преобразовать в datetime для дальнейшего анализа временных рядов
- **Modified_Time**: object - преобразовать в datetime для дальнейшего анализа временных рядов
- **Contact Owner Name - Manager**: object - преобразовать в string.

In [7]:
# убираю пробелы и спецсимволы в названиях колонок.

df_contacts.columns = (
    df_contacts.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
)

In [8]:
# первый и второй столбец переименовываю в 'Contact_ID' и 'Manager'

df_contacts = df_contacts.rename(columns={
    df_contacts.columns[0]: "Contact_ID",
    df_contacts.columns[1]: "Manager"
})

In [9]:
# удаляю записи, где в поле 'Manager' указано значение 'False' - такое поле одно

df_contacts = df_contacts[df_contacts["Manager"] != False]

In [10]:
# преобразование колонок 'Created_Time' и 'Modified_Time' к формату даты-времени 

df_contacts["Created_Time"] = pd.to_datetime(df_contacts["Created_Time"], errors="coerce", dayfirst=True)

df_contacts["Modified_Time"] = pd.to_datetime(df_contacts["Modified_Time"], errors="coerce", dayfirst=True)

In [11]:
# проверка преобразования  

print(df_contacts[["Created_Time", "Modified_Time"]].head(10))
print(df_contacts[["Created_Time", "Modified_Time"]].dtypes)

         Created_Time       Modified_Time
0 2023-06-27 11:28:00 2023-12-22 13:34:00
1 2023-07-03 11:31:00 2024-05-21 10:23:00
2 2023-07-02 22:37:00 2023-12-21 13:17:00
3 2023-07-03 05:44:00 2023-12-29 15:20:00
4 2023-07-04 10:11:00 2024-04-16 16:14:00
5 2023-07-04 12:57:00 2023-07-17 19:43:00
6 2023-07-03 20:17:00 2023-10-05 10:44:00
7 2023-07-04 15:40:00 2024-06-11 18:40:00
8 2023-07-04 22:03:00 2023-07-17 19:43:00
9 2023-07-03 20:39:00 2024-06-18 10:10:00
Created_Time     datetime64[ns]
Modified_Time    datetime64[ns]
dtype: object


In [12]:
# считаю количество пропусков по каждому столбцу

missing_values = df_contacts.isnull().sum()

print(missing_values)

Contact_ID       0
Manager          0
Created_Time     0
Modified_Time    0
dtype: int64


In [13]:
# преобразование колонок 'Manager' и 'Contact_ID' к строковому типу 

df_contacts["Manager"] = df_contacts["Manager"].astype("string")

df_contacts["Contact_ID"] = df_contacts["Contact_ID"].astype("string")

In [14]:
# проверяю количество дубликатов по строкам

duplicates_count = df_contacts.duplicated().sum()
print("Найдено дубликатов по строкам:", duplicates_count)

# и если есть, вывожу сами дубликаты

duplicates_rows = df_contacts[df_contacts.duplicated()]
display(duplicates_rows)


Найдено дубликатов по строкам: 0


,Contact_ID,Manager,Created_Time,Modified_Time


In [15]:
# проверка преобразования таблица Contacts

print("Типы данных:\n", df_contacts.dtypes)

Типы данных:
 Contact_ID       string[python]
Manager          string[python]
Created_Time     datetime64[ns]
Modified_Time    datetime64[ns]
dtype: object


### 1.2. Анализ датасета Calls

In [16]:
# базовая проверка

print("Типы данных:\n", df_calls.dtypes)
print("\nПропуски по столбцам:\n", df_calls.isna().sum())
print("\nДубликаты строк:", df_calls.duplicated().sum())

Типы данных:
 Id                              int64
Call Start Time                object
Call Owner Name                object
CONTACTID                     float64
Call Type                      object
Call Duration (in seconds)    float64
Call Status                    object
Dialled Number                float64
Outgoing Call Status           object
Scheduled in CRM              float64
Tag                           float64
dtype: object

Пропуски по столбцам:
 Id                                0
Call Start Time                   0
Call Owner Name                   0
CONTACTID                      3933
Call Type                         0
Call Duration (in seconds)       83
Call Status                       0
Dialled Number                95874
Outgoing Call Status           8999
Scheduled in CRM               8999
Tag                           95874
dtype: int64

Дубликаты строк: 0


#### **Анализ и предложения по типам данных:**

- **Id - Call_ID**: object - преобразовать в string.
- **Call Start Time - Call_Start_Time**: object - преобразовать в datetime для дальнейшего анализа временных рядов
- **Call Owner Name - Manager**: object - преобразовать в string.
- **CONTACTID - Contact_ID**: float64 - преобразовать в string.
- **Call Type**: object - преобразовать в категориальный тип данных.
- **Call Duration (in seconds) - Call_Duration_sec**: float64 - преобразовать в int64, т.к. это целые числа
- **Call Status - Call_Status**: object - преобразовать в категориальный тип данных.
- **Outgoing Call Status - Outgoing_Call_Status**: object - преобразовать в категориальный тип данных.
- **Scheduled in CRM - Scheduled_in_CRM**: float64 преобразовать в категориальный тип данных, чтобы эффективнее хранить и обрабатывать значения, так как они представляют собой бинарные состояния (запланировано или нет)

In [17]:
# переименование столбцов

df_calls = df_calls.rename(columns={
    "Id": "Call_ID",
    "Call Start Time": "Call_Start_Time",
    "Call Owner Name": "Manager",
    "CONTACTID": "Contact_ID",
    "Call Duration (in seconds)": "Call_Duration_sec",
    "Call Status": "Call_Status",
    "Outgoing Call Status": "Outgoing_Call_Status",
    "Scheduled in CRM": "Scheduled_in_CRM",
    "Tag": "Tag",
    "Dialled Number": "Dialled_Number" 
})

In [18]:
# удаляю полностью пустые столбцы

df_calls = df_calls.drop(columns=["Tag", "Dialled_Number"], errors="ignore")

In [19]:
# обрабатываю пропуски в столбцах (категориальные - Unknown, затем category)

df_calls["Contact_ID"] = df_calls["Contact_ID"].fillna("Unknown").astype("string")
df_calls["Call_Status"] = df_calls["Call_Status"].fillna("Unknown").astype("category")
df_calls["Outgoing_Call_Status"] = df_calls["Outgoing_Call_Status"].fillna("Unknown").astype("category")
df_calls["Scheduled_in_CRM"] = df_calls["Scheduled_in_CRM"].fillna("Unknown").astype("category")

In [20]:
# работаю со столбцом 'Call_Duration_sec'

if "Call_Duration_sec" in df_calls.columns:
    df_calls["Call_Duration_sec"] = df_calls["Call_Duration_sec"].astype("Float64") # преобразовываю в float, чтобы безопасно работать с NaN
    median_by_manager = df_calls.groupby("Manager")["Call_Duration_sec"].transform("median") 
    df_calls["Call_Duration_sec"] = df_calls["Call_Duration_sec"].fillna(median_by_manager) 
    global_median = df_calls["Call_Duration_sec"].median() # если вдруг у менеджера все звонки NaN, тогда глобальная медиана
    df_calls["Call_Duration_sec"] = df_calls["Call_Duration_sec"].fillna(global_median)
    df_calls["Call_Duration_sec"] = df_calls["Call_Duration_sec"].astype(int) # привожу к целому (длительность в секундах)

In [21]:
# преобразовываю типы

# Call_ID и Contact_ID в строки (уникальные ключи)
df_calls["Call_ID"] = df_calls["Call_ID"].astype("string")

# Call_Start_Time в datetime с явным форматом
df_calls["Call_Start_Time"] = pd.to_datetime(
    df_calls["Call_Start_Time"], 
    format="%d.%m.%Y %H:%M",  # день.месяц.год часы:минуты
    errors="coerce"
)

# Call_Type в category
if "Call Type" in df_calls.columns:
    df_calls["Call Type"] = df_calls["Call Type"].fillna("Unknown").astype("category")

# Manager в string
df_calls["Manager"] = df_calls["Manager"].astype("string")

In [22]:
# проверяю выбросы по длительности звонков

outliers = df_calls[df_calls["Call_Duration_sec"] > 50000]
print("Звонков с аномальной длительностью:", len(outliers))

Звонков с аномальной длительностью: 0


In [23]:
# финальная проверка

print("\nФинальные типы данных:\n", df_calls.dtypes)

# вывожу первые 5 строк таблицы
print("=== Первые строки df_calls ===")
print(df_calls.head())

# считаем мтоговое количество пропусков по каждому столбцу
missing_values_calls = df_calls.isnull().sum()
print(missing_values_calls)


Финальные типы данных:
 Call_ID                 string[python]
Call_Start_Time         datetime64[ns]
Manager                 string[python]
Contact_ID              string[python]
Call Type                     category
Call_Duration_sec                int64
Call_Status                   category
Outgoing_Call_Status          category
Scheduled_in_CRM              category
dtype: object
=== Первые строки df_calls ===
               Call_ID     Call_Start_Time   Manager             Contact_ID  \
0  5805028000000805001 2023-06-30 08:43:00  John Doe                Unknown   
1  5805028000000768006 2023-06-30 08:46:00  John Doe                Unknown   
2  5805028000000764027 2023-06-30 08:59:00  John Doe                Unknown   
3  5805028000000787003 2023-06-30 09:20:00  John Doe  5.805028000000645e+18   
4  5805028000000768019 2023-06-30 09:30:00  John Doe  5.805028000000645e+18   

  Call Type  Call_Duration_sec       Call_Status Outgoing_Call_Status  \
0   Inbound                171 

### 1.3 Анализ датасета Spend

In [24]:
# базовая проверка

print("Типы данных:\n", df_spend.dtypes)
print("\nПропуски по столбцам:\n", df_spend.isna().sum())

Типы данных:
 Date           datetime64[ns]
Source                 object
Campaign               object
Impressions             int64
Spend                 float64
Clicks                  int64
AdGroup                object
Ad                     object
dtype: object

Пропуски по столбцам:
 Date              0
Source            0
Campaign       5994
Impressions       0
Spend             0
Clicks            0
AdGroup        6828
Ad             6828
dtype: int64


#### **Анализ и предложения по типам данных:**

- **Source**: object - преобразовать в категориальный тип данных.
- **Campaign**: object - преобразовать в категориальный тип данных.
- **Impressions**: int64 - преобразовать в Int64.
- **AdGroup**: object - преобразовать в категориальный тип данных.
- **Ad**: object - преобразовать в категориальный тип данных.

**Дополнительно:** Были введены новые столбцы и правила, а также преобразование столбцов в категориальный тип данных выполненно с целью уменьшения объема памяти и ускорения операций группировки.

In [25]:
# проверка на дубликаты (найдено 917), после чего удаление

duplicates = df_spend.duplicated().sum()
print("\nДубликаты строк:", duplicates)
if duplicates > 0:
    df_spend = df_spend.drop_duplicates()


Дубликаты строк: 917


In [26]:
# переименование столбцов

df_spend = df_spend.rename(columns={
    "Date": "Date",
    "Source": "Source",
    "Campaign": "Campaign",
    "Impressions": "Impressions",
    "Spend": "Spend",
    "Clicks": "Clicks",
    "AdGroup": "Ad_Group",
    "Ad": "Ad"
})

In [27]:
# обработка пропусков (cтроковые на Unknown) 

for col in ["Source", "Campaign", "Ad_Group", "Ad"]:
    df_spend[col] = df_spend[col].fillna("Unknown")

In [28]:
# преобразование типов

df_spend["Date"] = pd.to_datetime(df_spend["Date"], errors="coerce") 
df_spend["Source"] = df_spend["Source"].astype("category")
df_spend["Campaign"] = df_spend["Campaign"].astype("category")
df_spend["Ad_Group"] = df_spend["Ad_Group"].astype("category")
df_spend["Ad"] = df_spend["Ad"].astype("category")

df_spend["Impressions"] = pd.to_numeric(df_spend["Impressions"], errors="coerce").astype("Int64")
df_spend["Spend"] = pd.to_numeric(df_spend["Spend"], errors="coerce").astype(float)
df_spend["Clicks"] = pd.to_numeric(df_spend["Clicks"], errors="coerce").astype("Int64")

In [29]:
# проверка выбросов 

# - отрицательные расходы
neg_spend = df_spend[df_spend["Spend"] < 0]
print("Отрицательных расходов:", len(neg_spend))

# - кликов больше, чем показов
df_spend["Clicks_Anomaly"] = df_spend["Clicks"] > df_spend["Impressions"]
print("Кликов > показов:", df_spend["Clicks_Anomaly"].sum())

Отрицательных расходов: 0
Кликов > показов: 1363


In [30]:
# метрики эффективности

df_spend["CTR"] = (df_spend["Clicks"] / df_spend["Impressions"].replace(0, pd.NA)) * 100
df_spend["CPC"] = df_spend["Spend"] / df_spend["Clicks"].replace(0, pd.NA)
df_spend["CPM"] = (df_spend["Spend"] / df_spend["Impressions"].replace(0, pd.NA)) * 1000

print("\nСводка по метрикам")
print("Средний CTR: ", df_spend["CTR"].mean())
print("Средний CPC: ", df_spend["CPC"].mean())
print("Средний CPM: ", df_spend["CPM"].mean())


Сводка по метрикам
Средний CTR:  1.7399564931186984
Средний CPC:  0.681936917219153
Средний CPM:  15.441618384088311


In [31]:
# топ источников по расходам

print("\nТоп источников по расходам:")
print(df_spend.groupby("Source", observed=True)["Spend"].sum().sort_values(ascending=False).head())
print(df_spend.groupby("Campaign", observed=True)["Clicks"].sum().sort_values(ascending=False).head())


Топ источников по расходам:
Source
Google Ads      57798.60
Facebook Ads    33754.72
Youtube Ads     14633.33
Bloggers        13439.00
Tiktok Ads      11985.67
Name: Spend, dtype: float64
Campaign
performancemax_eng_DE    160175
Unknown                  110057
youtube_shorts_DE         57873
discovery_DE              56889
12.07.2023wide_DE         22768
Name: Clicks, dtype: Int64


In [32]:
# динамика по дням

print("\nДинамика по дням")
print(df_spend.groupby("Date", observed=True)[["Spend", "Impressions", "Clicks"]].sum().head())


Динамика по дням
            Spend  Impressions  Clicks
Date                                  
2023-07-03  10.27          664      74
2023-07-04  48.19         7008     108
2023-07-05  96.14        21092     443
2023-07-06  87.13        27971     491
2023-07-07  91.83         7918     251


In [33]:
# топ Ad_Group по CTR

print("\nТоп Ad_Group по CTR:")
print(df_spend.groupby("Ad_Group", observed=True)["CTR"].mean().sort_values(ascending=False).head())


Топ Ad_Group по CTR:
Ad_Group
Unknown                  5.094951
accountant_wide          4.696851
berlin_wide              2.335522
interest_work            1.944995
wide_python-developer    1.810633
Name: CTR, dtype: Float64


In [34]:
# общие показатели

print("\nОбщий Spend:", df_spend["Spend"].sum())
print("Общий Clicks:", df_spend["Clicks"].sum())
print("Общий Impressions:", df_spend["Impressions"].sum())


Общий Spend: 149523.44999999998
Общий Clicks: 498455
Общий Impressions: 51079010


In [35]:
# подозрительные строки

print("\nСтроки с расходами, но без показов:")
print(df_spend[(df_spend["Spend"] > 0) & (df_spend["Impressions"] == 0)].head())

print("\nСтроки с показами, но без кликов:")
print(df_spend[(df_spend["Impressions"] > 0) & (df_spend["Clicks"] == 0)].head())


Строки с расходами, но без показов:
           Date Source Campaign  Impressions  Spend  Clicks Ad_Group       Ad  \
1585 2023-08-08    SMM  Unknown            0  100.0      30  Unknown  Unknown   
4074 2023-09-23    SMM  Unknown            0  250.0      21  Unknown  Unknown   
6835 2023-11-18    SMM  Unknown            0  110.0      13  Unknown  Unknown   
7898 2023-12-12    SMM  Unknown            0  165.0      27  Unknown  Unknown   
8485 2023-12-22    SMM  Unknown            0   50.0      22  Unknown  Unknown   

      Clicks_Anomaly   CTR        CPC   CPM  
1585            True  <NA>   3.333333  <NA>  
4074            True  <NA>  11.904762  <NA>  
6835            True  <NA>   8.461538  <NA>  
7898            True  <NA>   6.111111  <NA>  
8485            True  <NA>   2.272727  <NA>  

Строки с показами, но без кликов:
         Date        Source         Campaign  Impressions  Spend  Clicks  \
0  2023-07-03    Google Ads   gen_analyst_DE            6   0.00       0   
9  2023-07-03

In [36]:
# cброс индекса

df_spend.reset_index(drop=True, inplace=True)

In [37]:
# финальная проверка
print("\nФинальные типы данных:\n", df_spend.dtypes)

# первые 5 строк таблицы
print("=== Первые строки df_spend ===")
print(df_spend.head())

# количество пропусков по каждому столбцу
missing_values_calls = df_spend.isnull().sum()
print(missing_values_calls)


Финальные типы данных:
 Date              datetime64[ns]
Source                  category
Campaign                category
Impressions                Int64
Spend                    float64
Clicks                     Int64
Ad_Group                category
Ad                      category
Clicks_Anomaly           boolean
CTR                      Float64
CPC                      Float64
CPM                      Float64
dtype: object
=== Первые строки df_spend ===
        Date        Source               Campaign  Impressions  Spend  Clicks  \
0 2023-07-03    Google Ads         gen_analyst_DE            6   0.00       0   
1 2023-07-03    Google Ads  performancemax_eng_DE            4   0.01       1   
2 2023-07-03  Facebook Ads                Unknown            0   0.00       0   
3 2023-07-03    Google Ads                Unknown            0   0.00       0   
4 2023-07-03           CRM                Unknown            0   0.00       0   

  Ad_Group       Ad  Clicks_Anomaly   CTR   CPC

### 1.4 Анализ датасета Deals

In [38]:
# базовая проверка

print("Типы данных:\n", df_deals.dtypes)
print("\nПропуски по столбцам:\n", df_deals.isna().sum())

dup_rows = df_deals.duplicated().sum()
print("\nДубликаты строк (до):", dup_rows)

Типы данных:
 Id                     float64
Deal Owner Name         object
Closing Date            object
Quality                 object
Stage                   object
Lost Reason             object
Page                    object
Campaign                object
SLA                     object
Content                 object
Term                    object
Source                  object
Payment Type            object
Product                 object
Education Type          object
Created Time            object
Course duration        float64
Months of study        float64
Initial Amount Paid     object
Offer Total Amount      object
Contact Name           float64
City                    object
Level of Deutsch        object
dtype: object

Пропуски по столбцам:
 Id                         2
Deal Owner Name           31
Closing Date            6950
Quality                 2255
Stage                      2
Lost Reason             5471
Page                       2
Campaign                5528
SLA

#### **Анализ и предложения по типам данных:***

- **Id - Deal_ID**: float64 - преобразовать в string.
- **Deal Owner Name - Manager_Name**: object - преобразовать в категориальный тип данных.
- **Closing Date - Closing_Date**: object - преобразовать в datetime для дальнейшего анализа временных рядов.
- **Quality**: object - преобразовать в категориальный тип данных.
- **Stage**: object - преобразовать в категориальный тип данных.
- **Lost Reason - Lost_Reason**: object - преобразовать в категориальный тип данных.
- **Page**: object - удаляю.
- **Campaign**: object - преобразовать в категориальный тип данных.
- **SLA - SLA_seconds**: преобразуем значения в int64.
- **Content**: object - преобразовать в категориальный тип данных.
- **Term**: object - преобразовать в категориальный тип данных.
- **Source**: object - преобразовать в категориальный тип данных.
- **Payment Type - Payment_Type**: object - удаляю.
- **Product**: object - преобразовать в категориальный тип данных.
- **Education Type - Education_Type**: object - преобразовать в категориальный тип данных.
- **Created Time - Created_Time**: object -преобразовать в datetime для дальнейшего анализа временных рядов.
- **Course duration - Course_duration**: float64.
- **Months of study - Months_of_study**: удаляю.
- **Initial Amount Paid - Initial_Amount_Paid**: object - преобразовать в float64 для анализа финансовых данных (есть нечисловые значения - обработать).
- **Offer Total Amount - Offer_Total_Amount**: object - преобразовать в float64 для анализа финансовых данных (есть нечисловые значения - обработать).
- **Contact Name - Contact_Name**: float64 - преобразовать в string.
- **City**: object - преобразовать в категориальный тип данных.
- **Deutsch_Level_Normalized**: преобразовать в категориальный тип данных.

**Дополнительно:** Были введены новые столбцы и правила, а также преобразование столбцов в категориальный тип данных выполненно с целью уменьшения объема памяти и ускорения операций группировки.

In [39]:
# переименование столбцов

rename_map = {
    "Id": "Deal_ID",
    "Deal Owner Name": "Manager_Name",
    "Closing Date": "Closing_Date",
    "Quality": "Quality",
    "Stage": "Stage",
    "Lost Reason": "Lost_Reason",
    "Page": "Page",
    "Campaign": "Campaign",
    "SLA": "SLA",
    "Content": "Content",
    "Term": "Term",
    "Source": "Source",
    "Payment Type": "Payment_Type",
    "Product": "Product",
    "Education Type": "Education_Type",
    "Created Time": "Created_Time",
    "Course duration": "Course_duration",
    "Months of study": "Months_of_study",
    "Initial Amount Paid": "Initial_Amount_Paid",
    "Offer Total Amount": "Offer_Total_Amount",
    "Contact Name": "Contact_Name",
    "City": "City",
    "Level of Deutsch": "Deutsch_Level_Raw"   
}
df_deals = df_deals.rename(columns={k: v for k, v in rename_map.items() if k in df_deals.columns})

In [40]:
# удаляю колонку 'Page'

if "Page" in df_deals.columns:
    df_deals = df_deals.drop(columns=["Page"])

In [41]:
# убираю строки без Manager_Name

df_deals = df_deals[~df_deals["Manager_Name"].isna()]

In [42]:
# убираю все пустые Contact_Name кроме одной строки (минимальный индекс)

na_contacts = df_deals[df_deals["Contact_Name"].isna()].index
if len(na_contacts) > 1:
    keep_idx = na_contacts.min()
    drop_idx = na_contacts.drop(keep_idx)
    df_deals = df_deals.drop(drop_idx)

In [43]:
# нормализация сумм Initial_Amount_Paid и Offer_Total_Amount

df_deals["Initial_Amount_Paid_Clean"] = (
    df_deals["Initial_Amount_Paid"]
    .astype(str).str.replace("€", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.replace(r"[^\d.]", "", regex=True)
    .replace("", np.nan)
    .astype("Float64")
)

df_deals["Offer_Total_Amount_Clean"] = (
    df_deals["Offer_Total_Amount"]
    .astype(str).str.replace("€", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .replace("nan", np.nan)
    .astype("Float64")
)

In [44]:
# если Initial > Offer → приравниваем

df_deals.loc[
    df_deals["Initial_Amount_Paid_Clean"] > df_deals["Offer_Total_Amount_Clean"],
    "Initial_Amount_Paid_Clean"
] = df_deals["Offer_Total_Amount_Clean"]

In [45]:
# флаг подарков (символическая оплата)

df_deals["Gift_Flag"] = df_deals["Initial_Amount_Paid_Clean"].isin([0, 1, 9])

In [46]:
# заполняю пропуски Offer_Total_Amount_Clean медианой по продукту

median_offer_by_product = df_deals.groupby("Product", observed=True)["Offer_Total_Amount_Clean"].transform("median")
df_deals["Offer_Total_Amount_Clean"] = df_deals["Offer_Total_Amount_Clean"].fillna(median_offer_by_product)

df_deals = df_deals.drop(columns=["Initial_Amount_Paid", "Offer_Total_Amount"])

In [47]:
# создаю новый столбец Payment_Status

df_deals["Payment_Status"] = "Unpaid"
df_deals.loc[
    (df_deals["Closing_Date"].notna()) & (df_deals["Stage"] == "Payment Done"),
    "Payment_Status"
] = "Paid"

df_deals["Payment_Status"] = "Unpaid"

df_deals.loc[
    (df_deals["Closing_Date"].notna()) &
    (df_deals["Stage"].str.casefold() == "payment done".casefold()),
    "Payment_Status"
] = "Paid"

df_deals["Payment_Status"].value_counts()

Payment_Status
Unpaid    20991
Paid        514
Name: count, dtype: int64

In [48]:
# SLA перевожу в секунды 

df_deals["SLA"] = df_deals["SLA"].replace("[]", np.nan)

df_deals["SLA_seconds"] = pd.to_timedelta(
    df_deals["SLA"].astype(str), errors="coerce"
).dt.total_seconds()

mask_num = pd.to_numeric(df_deals["SLA"], errors="coerce").notna()
df_deals.loc[mask_num, "SLA_seconds"] = df_deals.loc[mask_num, "SLA"].astype(float) * 86400

# заполненяю пропуски SLA медианой по менеджеру
median_sla_per_manager = df_deals.groupby("Manager_Name")["SLA_seconds"].transform("median")
df_deals["SLA_seconds"] = (
    df_deals["SLA_seconds"].fillna(median_sla_per_manager).infer_objects(copy=False)
)

df_deals["SLA_seconds"] = (
    pd.to_numeric(df_deals["SLA_seconds"], errors="coerce")
      .round()                
      .astype("Int64")       
)

In [49]:
# Course_duration заполняю медианой по продукту

median_by_product = df_deals.groupby("Product", observed=True)["Course_duration"].transform("median")
df_deals["Course_duration"] = df_deals["Course_duration"].fillna(median_by_product)

In [50]:
# убираю дубликаты Lost 

df_deals = df_deals.drop(
    df_deals[(df_deals["Stage"] == "Lost") & (df_deals["Lost_Reason"] == "Duplicate")].index
)

In [51]:
# очистка уровней немецкого

LEVELS = ["A0","A1","A2","B1","B2","C1","C2"]
RANK = {lvl: i for i, lvl in enumerate(LEVELS)}
TRANS = str.maketrans({"а":"a","А":"A","в":"b","В":"B","б":"b","Б":"B","с":"c","С":"C"})
C1_TRIGGERS = ("граждан", "живу", "живем", "живём", "живет", "живёт")
GER_MARKERS = ("ня", "нем", "de", "deutsch", "german")
ENG_MARKERS = ("ая", "англ", "english")

def normalize_level_cell(x) -> str:
    if pd.isna(x) or not str(x).strip():
        return "Unknown"
    raw = str(x); s = raw.lower()
    s = re.sub(r"\bf\s*([0-2])\b", r"b\1", s)
    s = s.translate(TRANS)
    candidates = []
    for m in re.finditer(r"([abc])\s*([0-2])", s):
        lvl = f"{m.group(1).upper()}{m.group(2)}"
        a, b = m.span()
        window = s[max(0,a-15):a] + " " + s[b:b+15]
        if any(k in window for k in ENG_MARKERS) and not any(k in window for k in GER_MARKERS):
            continue
        candidates.append(lvl)
    if candidates:
        return max(candidates, key=lambda z: RANK.get(z,-1))
    for m in re.finditer(r"\b([abc])\b", s):
        window = s[max(0,m.start()-15):m.start()] + " " + s[m.end():m.end()+15]
        if any(k in window for k in ENG_MARKERS) and not any(k in window for k in GER_MARKERS):
            continue
        return f"{m.group(1).upper()}1"
    if any(k in s for k in C1_TRIGGERS):
        return "C1"
    return "Unknown"

df_deals["Deutsch_Level_Normalized"] = df_deals["Deutsch_Level_Raw"].apply(normalize_level_cell)

In [52]:
# заполняю Education_Type

edu, dur, total, prod = "Education_Type", "Course_duration", "Offer_Total_Amount_Clean", "Product"
s = df_deals[edu].astype("string").str.strip().str.lower()
mask = df_deals[[dur, total, prod]].notna().all(axis=1)
group_mode = (
    df_deals.loc[mask]
         .groupby([dur, total])[edu]
         .transform(lambda s: s.mode(dropna=True).iloc[0] if s.notna().any() else pd.NA)
)
df_deals.loc[mask, edu] = df_deals.loc[mask, edu].fillna(group_mode)
df_deals[edu] = df_deals[edu].fillna("Unknown")


# Доп. lookup (Product + Offer_Total_Amount_Clean → Education_Type)
lookup = (
    df_deals.dropna(subset=["Education_Type"])
         .drop_duplicates(subset=["Product", "Offer_Total_Amount_Clean"])
         .set_index(["Product", "Offer_Total_Amount_Clean"])["Education_Type"]
         .to_dict()
)
df_deals["Education_Type"] = df_deals.apply(
    lambda row: lookup.get((row["Product"], row["Offer_Total_Amount_Clean"]), row["Education_Type"]),
    axis=1
)

In [53]:
# работа с City

df_deals["City"] = (
    df_deals["City"]
    .replace("-", pd.NA)
    .fillna(df_deals.groupby("Contact_Name")["City"].transform(lambda s: s.dropna().iloc[0] if not s.dropna().empty else pd.NA))
    .fillna("Unknown")
    .astype(str)
    .str.strip()                        
    .str.replace("nan", "Unknown")     
    .str.replace("unknown", "Unknown")  
    .replace({"Villingen\u2011Schwenningen": "Villingen-Schwenningen"})
    .astype("category")
)

In [54]:
# даты: Created/Closing (Closing переводим на конец дня)
if "Created_Time" in df_deals:
    df_deals["Created_Time"] = pd.to_datetime(df_deals["Created_Time"], errors="coerce", dayfirst=True)
    
df_deals["Closing_Date"] = pd.to_datetime(
    df_deals["Closing_Date"], format="%d.%m.%Y", errors="coerce"
) + pd.Timedelta(hours=23, minutes=59, seconds=59)

In [55]:
# удаляю колонки (Payment_Type т.к. поля Payment Type заполняется менеджерами вручную, 
# и в них могут быть ошибки типа неправильного статуса (например, может быть указан one payment, хотя
# клиент платит в рассрочку). Причем определить, в чем именно ошибка, в случае такие расхождений невозможно)

df_deals = df_deals.drop(columns=["Payment_Type", "SLA", "Months_of_study", "Deutsch_Level_Raw"])

In [56]:
# заполняю пропуски в категориальных полях Unknown

for col in ["Quality", "Lost_Reason", "Campaign", "Content", "Term", "Product"]:
    if col in df_deals.columns:
        df_deals[col] = df_deals[col].fillna("Unknown")

In [57]:
# финальное приведение типов

df_deals = df_deals.astype({
    "Deal_ID": "string",            
    "Contact_Name": "string",       
    "Manager_Name": "category",
    "Quality": "category",
    "Stage": "category",
    "Lost_Reason": "category",
    "Campaign": "category",
    "Content": "category",
    "Term": "category",
    "Source": "category",
    "Product": "category",
    "Education_Type": "category",
    "Payment_Status": "category",
    "SLA_seconds": "Int64",          
    "Deutsch_Level_Normalized": "category"
})

In [58]:
# считаю и удаляю полные дубликаты

before = len(df_deals)
dup_to_remove = df_deals.duplicated(keep="first").sum()
print(f"Полных дублей к удалению: {dup_to_remove}")

df_deals = df_deals.drop_duplicates(keep="first").reset_index(drop=True)
removed = before - len(df_deals)
print(f"Удалено полных дублей: {removed}")

Полных дублей к удалению: 2
Удалено полных дублей: 2


In [59]:
# cброс индекса

df_deals.reset_index(drop=True, inplace=True)

In [60]:
# финальная проверка
print("Итоговый размер:", df_deals.shape)
print("Пустых менеджеров:", df_deals["Manager_Name"].isna().sum())
print("Пустых контактов:", df_deals["Contact_Name"].isna().sum())


# первые 5 строк таблицы
print("=== Первые строки df_deals ===")
print(df_deals.head())

# количество пропусков по каждому столбцу
missing_values_deals = df_deals.isnull().sum()
print(missing_values_deals)
print("Типы данных:\n", df_deals.dtypes)

Итоговый размер: (19771, 22)
Пустых менеджеров: 0
Пустых контактов: 1
=== Первые строки df_deals ===
                 Deal_ID   Manager_Name        Closing_Date  \
0  5.805028000056865e+18       Ben Hall                 NaT   
1   5.80502800005686e+18  Ulysses Adams                 NaT   
2  5.805028000056832e+18  Ulysses Adams 2024-06-21 23:59:59   
3  5.805028000056824e+18       Eva Kent 2024-06-21 23:59:59   
4  5.805028000056874e+18       Ben Hall 2024-06-21 23:59:59   

             Quality     Stage     Lost_Reason                  Campaign  \
0            Unknown  New Lead         Unknown             03.07.23women   
1            Unknown  New Lead         Unknown                   Unknown   
2     D - Non Target      Lost      Non target                engwien_AT   
3  E - Non Qualified      Lost  Invalid number  04.07.23recentlymoved_DE   
4     D - Non Target      Lost      Non target              discovery_DE   

              Content           Term          Source  ...      

### Сохранение очищенных датафреймов

In [61]:
import pickle

# Сохраняем несколько датафреймов сразу
data = [df_calls, df_contacts, df_deals, df_spend]

with open("data.pickle", "wb") as f:
    pickle.dump(data, f)

print("Датафреймы сохранены в data.pickle")


Датафреймы сохранены в data.pickle
